Contruimos el pipeline completo que siga el siguiente flujo:

RAW DATA
↓
Cleaning
↓
Feature Engineering
↓
Train/Test Split
↓
SMOTE
↓
Preprocessing
↓
XGBoost
↓
Guardar pipeline completo

# Imports

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier

from imblearn.over_sampling import SMOTE

# Cargamos el data set - raw

In [7]:
DATA_PATH = Path(
    "../data/raw/01-hotel_bookings.csv"
)

df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(119390, 32)


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,01-07-15
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,01-07-15
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,02-07-15
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,02-07-15
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,03-07-15


In [8]:
df.columns

Index(['hotel', 'is_canceled', 'lead_time', 'arrival_date_year',
       'arrival_date_month', 'arrival_date_week_number',
       'arrival_date_day_of_month', 'stays_in_weekend_nights',
       'stays_in_week_nights', 'adults', 'children', 'babies', 'meal',
       'country', 'market_segment', 'distribution_channel',
       'is_repeated_guest', 'previous_cancellations',
       'previous_bookings_not_canceled', 'reserved_room_type',
       'assigned_room_type', 'booking_changes', 'deposit_type', 'agent',
       'company', 'days_in_waiting_list', 'customer_type', 'adr',
       'required_car_parking_spaces', 'total_of_special_requests',
       'reservation_status', 'reservation_status_date'],
      dtype='str')

# Data cleaning

In [9]:
# Eliminar columnas leakage
cols_drop = [
    "reservation_status",
    "reservation_status_date",
    "agent",
    "company",
    "country"
]

df.drop(columns=cols_drop, inplace=True)

# Eliminar reservas sin personas
mask_people = (
    (df["adults"] == 0) &
    (df["children"].fillna(0) == 0) &
    (df["babies"] == 0)
)

df = df[~mask_people]

# Eliminar nulos en children
df.dropna(subset=["children"], inplace=True)

# Corregir tipo
df["children"] = df["children"].astype(int)

# Eliminar children=10
df = df[df["children"] != 10]

# Eliminar adr negativo
df = df[df["adr"] >= 0]

# Capear adr
p99 = df["adr"].quantile(0.99)

df["adr"] = df["adr"].clip(upper=p99)

print(df.shape)

(119204, 27)


## FEATURE ENGINEERING

In [10]:
# Total nights
df["total_nights"] = (
    df["stays_in_week_nights"] +
    df["stays_in_weekend_nights"]
)

# Total guests
df["total_guests"] = (
    df["adults"] +
    df["children"] +
    df["babies"]
)

# High season
high_season = ["July", "August"]

df["is_high_season"] = (
    df["arrival_date_month"]
    .isin(high_season)
    .astype(int)
)

# Log transforms
df["adr_log"] = np.log1p(df["adr"])
df["lead_time_log"] = np.log1p(df["lead_time"])

## ELIMINAR VARIABLES REDUNDANTES

In [11]:
df.drop(columns=[
    "adults",
    "children",
    "babies",
    "adr",
    "lead_time",
    "stays_in_week_nights",
    "stays_in_weekend_nights"
], inplace=True)

## SPLIT

In [12]:
X = df.drop("is_canceled", axis=1)
y = df["is_canceled"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

## COLUMNAS

In [13]:
num_cols = X_train.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

cat_cols = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

print(num_cols)
print(cat_cols)

['arrival_date_year', 'arrival_date_week_number', 'arrival_date_day_of_month', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'days_in_waiting_list', 'required_car_parking_spaces', 'total_of_special_requests', 'total_nights', 'total_guests', 'is_high_season', 'adr_log', 'lead_time_log']
['hotel', 'arrival_date_month', 'meal', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']


C:\Users\Luz Maria\AppData\Local\Temp\ipykernel_17220\4184047617.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X_train.select_dtypes(


## PREPROCESSOR

In [14]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols)
])

## PREPROCESS TRAIN

In [15]:
X_train_t = preprocessor.fit_transform(X_train)

X_test_t = preprocessor.transform(X_test)

## SMOTE

In [16]:
smote = SMOTE(random_state=42)

X_train_bal, y_train_bal = smote.fit_resample(
    X_train_t,
    y_train
)

print(X_train_bal.shape)

(120016, 73)


## MODELO FINAL

In [17]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=10,
    learning_rate=0.2,
    colsample_bytree=0.7,
    eval_metric="logloss",
    random_state=42
)

## FIT

In [18]:
xgb_model.fit(
    X_train_bal,
    y_train_bal
)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.7
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

## EVALUACIÓN

In [19]:
y_pred = xgb_model.predict(X_test_t)

y_proba = xgb_model.predict_proba(X_test_t)[:,1]

print("Accuracy:",
      accuracy_score(y_test, y_pred))

print("Recall:",
      recall_score(y_test, y_pred))

print("F1:",
      f1_score(y_test, y_pred))

print("AUC:",
      roc_auc_score(y_test, y_proba))

Accuracy: 0.8689652279686255
Recall: 0.7957913791152845
F1: 0.8182875756165658
AUC: 0.9357536881721197


## PIPELINE COMPLETO REAL

In [22]:
from sklearn.pipeline import Pipeline

full_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("clf", xgb_model)
])

## GUARDAR

In [23]:
import joblib

joblib.dump(
    full_pipeline,
    "../models/final_pipeline.pkl"
)

['../models/final_pipeline.pkl']